# Exercices XP : MCP minimal via STDIO (Version Étudiant )

Nous allons construire un petit serveur MCP et un client MCP qui communiquent via **STDIO** (l'entrée/sortie standard).

**Important** : ce code est prévu pour être exécuté dans un notebook Jupyter **local** (sur votre ordinateur), pas dans Colab, car il a besoin de lancer des sous-processus (le serveur) depuis le client.

## Ce que vous allez apprendre
- Comment MCP (Model Context Protocol) organise les **hôtes / clients / serveurs**, et pourquoi le transport STDIO est idéal en local.
- Comment déclarer un **outil** (tool = une action que le client peut exécuter) sur un serveur.
- Comment déclarer une **ressource** (resource = une donnée en lecture seule) sur un serveur.
- Comment écrire un client qui s'initialise, liste les fonctionnalités disponibles, puis les utilise.

## Installation
Exécutez la cellule d'installation ci-dessous. Python 3.10 ou plus récent est requis.

In [ ]:
# On installe le SDK MCP ainsi que son outil en ligne de commande (CLI)
# Le "[cli]" ajoute la commande `mcp` utilisable dans le terminal
%pip install -qU "mcp[cli]"

In [ ]:
# Vérification rapide : on affiche la version de Python installée...
!python --version
# ... et on vérifie que la commande "mcp" est bien disponible
!mcp --help | head -n 5

## A. Le serveur (server.py)

Nous créons un petit serveur MCP nommé `"Demo"` qui expose :
- Un **outil** `add(a, b)` qui retourne la somme de deux entiers.
- Une **ressource** `greeting://{name}` qui retourne `"Hello, {name}!"`.
- Une boucle principale qui démarre le serveur via STDIO.

In [ ]:
%%writefile server.py
# ============================================================
# server.py
# ------------------------------------------------------------
# Ce fichier définit un serveur MCP (Model Context Protocol) minimal.
# Un serveur MCP expose :
#   - des "outils" (tools)     -> des fonctions/actions que le client peut appeler
#   - des "ressources"          -> des données en lecture seule accessibles via une URI
# ============================================================

from mcp.server.fastmcp import FastMCP

# On crée une instance de serveur MCP.
# "Demo" est simplement le nom que l'on donne à notre serveur.
mcp = FastMCP("Demo")


# ------------------------------------------------------------
# OUTIL (TOOL)
# ------------------------------------------------------------
# Le décorateur @mcp.tool() enregistre automatiquement la fonction
# ci-dessous comme un "outil" disponible sur le serveur.
# Un client MCP pourra alors découvrir cet outil et l'appeler à distance.
@mcp.tool()
def add(a: int, b: int) -> int:
    """Retourne la somme de deux entiers."""
    # On additionne simplement les deux paramètres reçus et on retourne le résultat
    return a + b


# ------------------------------------------------------------
# RESSOURCE (RESOURCE)
# ------------------------------------------------------------
# Une ressource est une donnée en lecture seule, identifiée par une URI.
# Ici "greeting://{name}" est un "template d'URI" : la partie {name}
# est une variable qui sera remplacée par la valeur demandée par le client
# (par exemple : greeting://hello -> name = "hello")
@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Retourne un message de salutation pour le nom donné."""
    # On construit et on retourne la chaîne de salutation demandée
    return f"Hello, {name}!"


# ------------------------------------------------------------
# POINT D'ENTRÉE DU PROGRAMME
# ------------------------------------------------------------
if __name__ == "__main__":
    # On démarre la boucle du serveur en utilisant le transport STDIO.
    # "STDIO" signifie que le serveur communique via l'entrée/sortie standard
    # du processus (pratique pour un développement 100% local, sans réseau).
    mcp.run(transport="stdio")

## B. Le client (client.py)

Le client va :
1. Démarrer (spawn) le serveur via la commande `mcp` en utilisant le transport STDIO.
2. Initialiser une session MCP (`ClientSession`).
3. Lister les ressources puis les outils disponibles, et afficher leurs noms.
4. Lire la ressource `greeting://hello` et afficher son contenu.
5. Appeler l'outil `add` avec `a=1, b=7` et afficher le résultat.

In [ ]:
%%writefile client.py
# ============================================================
# client.py
# ------------------------------------------------------------
# Ce fichier définit un client MCP qui va :
#   1. Lancer automatiquement le serveur (server.py) via STDIO
#   2. Se connecter et initialiser une session
#   3. Découvrir (lister) les ressources et outils disponibles
#   4. Lire une ressource et appeler un outil, puis afficher les résultats
# ============================================================

import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# On définit comment lancer le serveur : la commande "mcp run server.py"
# sera exécutée automatiquement par le client (pas besoin d'ouvrir un 2e terminal).
server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


def extract_content(payload):
    """Fonction utilitaire : essaie d'extraire le texte utile depuis une réponse MCP."""
    # Certaines réponses (ex: lecture de ressource) contiennent une liste "contents"
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            # Le premier élément peut avoir un attribut ".text"
            if hasattr(first, "text"):
                return first.text
            # ...ou être un dictionnaire contenant la clé "text"
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            # Sinon on retourne sa représentation texte brute
            return str(first)
    # D'autres réponses (ex: appel d'outil) contiennent directement un attribut "content"
    if hasattr(payload, "content"):
        return payload.content
    # En dernier recours, on retourne simplement la représentation texte de l'objet
    return str(payload)


async def run():
    # On ouvre la connexion STDIO vers le serveur.
    # Le serveur (server.py) est lancé automatiquement en arrière-plan.
    async with stdio_client(server_params) as (read, write):
        # On crée une session client à partir des flux de lecture/écriture STDIO
        async with ClientSession(read, write) as session:
            # Étape obligatoire : on initialise la session (poignée de main avec le serveur)
            await session.initialize()

            # --------------------------------------------------------
            # 1) On liste les RESSOURCES disponibles sur le serveur
            # --------------------------------------------------------
            resources_result = await session.list_resources()
            print("Ressources disponibles :")
            for resource in resources_result.resources:
                print(f" - {resource.uri}")

            # --------------------------------------------------------
            # 2) On liste les OUTILS disponibles sur le serveur
            # --------------------------------------------------------
            tools_result = await session.list_tools()
            print("\nOutils disponibles :")
            for tool in tools_result.tools:
                print(f" - {tool.name}")

            # --------------------------------------------------------
            # 3) On lit la ressource "greeting://hello"
            # --------------------------------------------------------
            greeting_result = await session.read_resource("greeting://hello")
            print("\nContenu de greeting://hello :")
            print(extract_content(greeting_result))

            # --------------------------------------------------------
            # 4) On appelle l'outil "add" avec a=1 et b=7
            # --------------------------------------------------------
            add_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("\nRésultat de add(1, 7) :")
            print(extract_content(add_result))


if __name__ == "__main__":
    # On lance la fonction asynchrone run() dans une boucle d'événements asyncio
    asyncio.run(run())

## C. Exécution

**Option 1 — un seul terminal** (le client lance automatiquement le serveur) :
```bash
python client.py
```

**Option 2 — deux terminaux** (pour voir séparément les logs du serveur) :
```bash
# Terminal 1
mcp run server.py

# Terminal 2
python client.py
```

Ci-dessous, on exécute simplement `client.py`, qui se charge de démarrer le serveur automatiquement.

In [ ]:
# On exécute le client (qui va démarrer le serveur via STDIO automatiquement)
!python client.py

## Dépannage (Troubleshooting)

- `mcp : commande non trouvée` → vérifiez que votre environnement virtuel (venv) est bien activé, ou relancez la cellule d'installation.
- `Connection closed` → ouvrez un second terminal et lancez `mcp run server.py` pour voir les éventuelles erreurs de démarrage du serveur.
- Erreur de type (`type mismatch`) → vérifiez que les arguments JSON envoyés correspondent bien aux types attendus par la fonction (ici, `a` et `b` doivent être des entiers).